# 01 — Exploratory Data Analysis & Preprocessing

**Owner**: Noah  
**Course**: Machine Learning III (Unsupervised Learning) — Albert School  
**Dataset**: AI4I 2020 Predictive Maintenance (10 000 observations)

**Goal**: understand the data, identify cleaning needs, and design the preprocessing pipeline that will be consumed by the four anomaly detection models (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope).

**Key constraint**: the column `Machine failure` (and its subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`) must NOT be used as a training signal. We only inspect the failure rate to inform the `contamination` hyperparameter, then set the labels aside for the Part 4 final evaluation.

## 1. Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = Path.cwd().parent / "ai4i2020.csv"

## 2. Load the dataset

We load the CSV and look at the first rows to confirm the schema.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Schema and missing values

We expect 14 columns: 2 identifiers (`UDI`, `Product ID`), 1 categorical (`Type`), 5 numeric sensors, and 6 label columns (the global `Machine failure` flag plus 5 failure subtypes).

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

**Observation**: 10 000 rows × 14 columns, **zero missing values**. This is a clean industrial dataset, so we can skip imputation entirely. All numeric features are already in the right dtype (`float64` for temperatures and torque, `int64` for rotational speed and tool wear).

In [ ]:
df.describe()

## 4. Identify columns to keep, drop, and hold out

For unsupervised anomaly detection we must isolate three groups:

- **Identifiers** (`UDI`, `Product ID`): drop — no information for an anomaly model.
- **Held-out labels** (`Machine failure` + 5 failure subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`): drop from training. The 5 subtypes are essentially leaks of `Machine failure`. We keep `Machine failure` aside for the Part 4 "reveal".
- **Features** (`Type` + 5 numeric sensors): used for training.

In [ ]:
ID_COLUMNS = ["UDI", "Product ID"]
LABEL_COLUMNS = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
NUMERIC_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
CATEGORICAL_FEATURES = ["Type"]

y_true = df["Machine failure"].copy()
X = df.drop(columns=ID_COLUMNS + LABEL_COLUMNS)

print(f"X shape: {X.shape}")
print(f"X columns: {list(X.columns)}")
print(f"y_true held out: {y_true.shape[0]} rows")

## 5. Distribution of numeric features

We plot histograms and boxplots side by side for each of the 5 sensor variables. We are looking for: skewness (which threatens the Gaussian assumption of Elliptic Envelope), outliers (which could be the anomalies we want to detect), and scale differences (which justify standardization for distance-based models).

In [ ]:
fig, axes = plt.subplots(len(NUMERIC_FEATURES), 2, figsize=(12, 3 * len(NUMERIC_FEATURES)))
for i, col in enumerate(NUMERIC_FEATURES):
    sns.histplot(df[col], kde=True, ax=axes[i, 0])
    axes[i, 0].set_title(f"Histogram — {col}")
    sns.boxplot(x=df[col], ax=axes[i, 1])
    axes[i, 1].set_title(f"Boxplot — {col}")
plt.tight_layout()
plt.savefig("../outputs/figures/numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
skew_summary = df[NUMERIC_FEATURES].skew().round(3).rename("skewness").to_frame()
skew_summary["abs_skew"] = skew_summary["skewness"].abs()
skew_summary.sort_values("abs_skew", ascending=False)

**Takeaways from distributions**

- **Air temperature** and **Process temperature** look approximately symmetric (skew ≈ 0.1 and 0.0) — compatible with a Gaussian assumption.
- **Torque** is essentially symmetric (skew ≈ 0).
- **Tool wear** is uniform-like (skew ≈ 0), as expected for a wear-time variable that grows linearly.
- **Rotational speed** is strongly right-skewed (**skew ≈ 1.99**) with a long tail of high-speed values. **This violates the Gaussian assumption** of Elliptic Envelope and is a key fact for our modelling critique.
- The features live on very different scales: temperatures in the 295-305 K range, torque around 40 Nm, tool wear up to ~250 min, but rotational speed up to several thousand rpm. **Distance-based models will be dominated by `Rotational speed [rpm]` if we don't standardize.**

## 6. Distribution of the categorical feature `Type`

`Type` encodes the product variant: `L` = Low quality, `M` = Medium, `H` = High. We check the class balance and the failure rate per type — useful context for the business narrative.

In [ ]:
type_counts = df["Type"].value_counts()
type_share = df["Type"].value_counts(normalize=True).round(4)
type_summary = pd.concat([type_counts.rename("count"), type_share.rename("share")], axis=1)
type_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="Type", data=df, order=["L", "M", "H"], ax=ax)
ax.set_title("Distribution of product Type (L=Low, M=Medium, H=High)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../outputs/figures/type_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation**: 60 % of products are Low quality (`L`), 30 % Medium (`M`), 10 % High (`H`). Imbalanced but not extreme. For the distance-based models we will encode `Type` with one-hot encoding (see section 8 for the rationale).